Você sabia que no Databricks, até um simples delete de arquivo aparece como uma "query" no histórico?

Isso me te deixou curioso(a)? e a resposta é mais interessante do que parece.

Quando rodei um `dbutils.fs.rm()` para apagar um arquivo em um Volume do Unity Catalog, o Databricks registrou a operação como uma consulta, com tempo de execução, start time e end time.

**Mas por quê?**

O Databricks instrumenta praticamente tudo que roda em um notebook — seja SQL, Python ou operação de sistema de arquivos — como um evento rastreável. Isso acontece por alguns motivos:

- 📋 **Auditoria:** tudo fica registrado por usuário, cluster e horário
- 🔐 **Unity Catalog:** qualquer acesso a Volumes passa pela camada de governança
- 📊 **Interface unificada:** facilita monitorar o notebook inteiro em um só lugar

No meu caso, os 0 bytes lidos/escritos e 0 rows confirmaram que não houve processamento Spark de verdade — foi só uma operação de filesystem registrada pelo sistema de auditoria.

O Spark só otimiza com seu Catalyst Optimizer quando você trabalha com DataFrames, leituras de arquivos ou queries SQL. Um simples `dbutils.fs.rm()` não passa por esse motor.

Pequeno detalhe, grande aprendizado. 🚀

É esse tipo de curiosidade que transforma um engenheiro de dados em alguém que realmente entende a plataforma — não só usa.

---


Acompanhe o histórico de queries para identificar:
- Operações que consomem mais tempo ou recursos
- Possíveis gargalos de performance
- Comportamento de operações administrativas (como deletes, renomeios, etc.)

Entender o histórico é fundamental para otimizar e governar seu ambiente Databricks!

E se você pudesse perguntar aos seus dados em linguagem natural, por exemplo:  
**"Quais foram as 5 queries que mais demoraram para rodar nos últimos 5 dias?"**

Isso facilita a análise e torna o uso do Databricks ainda mais intuitivo!

Vamos conhecer mais a tabela de 
FROM 
  `system`.`query`.`history`

In [0]:
%sql
SELECT `statement_id`, `executed_by`, `statement_text`, `total_duration_ms`, `start_time`, `end_time`
FROM `system`.`query`.`history`
WHERE `start_time` >= current_timestamp() - interval 5 days
ORDER BY `total_duration_ms` DESC
LIMIT 5

In [0]:
%sql

select * from system.query.history limit 10

In [0]:
%sql
select * from system.query.history 
where statement_type = 'DROP'

In [0]:
%sql

--Análise de performance das queries dos últimos 7 dias
SELECT
  DATE(start_time) as data_execucao,
  executed_by,
  client_application,
  execution_status,
  COUNT(*) as total_queries,
  ROUND(AVG(total_duration_ms) / 1000, 2) as tempo_medio_segundos,
  ROUND(MAX(total_duration_ms) / 1000, 2) as tempo_maximo_segundos,
  ROUND(AVG(read_bytes) / 1024 / 1024, 2) as mb_lidos_medio,
  SUM(
    CASE
      WHEN from_result_cache THEN 1
      ELSE 0
    END
  ) as queries_do_cache
FROM
  system.query.history
WHERE
  start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY
  1,  2,  3,  4
ORDER BY
  1 DESC,  2
LIMIT 100;

outras infos de sistema

In [0]:
%sql
SHOW SCHEMAS IN system

In [0]:
%sql
show tables in system.billing

In [0]:
%sql
select * from system.billing.usage limit 10